In [7]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import warnings # Ignore specific warnings
warnings.filterwarnings("ignore")

In [8]:
df1 = pd.read_excel('output_lube_oil_g11.xlsx')

In [9]:
# انتخاب ستون‌ها برای استانداردسازی
data_to_scale = df1[['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287']]

# استانداردسازی داده‌ها
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_scale)

# تبدیل خروجی به دیتافریم با همان نام ستون‌ها
scaled_df = pd.DataFrame(scaled_data, columns=['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287'])


In [10]:
scaled_df_clean = scaled_df.dropna()

In [11]:
# اجرای DBSCAN
dbscan = DBSCAN(eps=0.7, min_samples=6)
labels = dbscan.fit_predict(scaled_df_clean)

# اضافه کردن لیبل‌ها به دیتافریم
df = scaled_df_clean.copy()
df['label'] = labels

# جدا کردن داده‌های نویز و خوشه‌ها
noise_mask = df['label'] == -1
cluster_mask = df['label'] != -1

noise_points = df[noise_mask].drop(columns='label').values
cluster_points = df[cluster_mask].drop(columns='label').values

# محاسبه فاصله هر نویز از نزدیک‌ترین نقطه در خوشه‌ها
distances = pairwise_distances(noise_points, cluster_points)
min_distances = distances.min(axis=1)

# نرمال‌سازی فاصله‌ها به بازه 0 تا 1
normalized_weights = (min_distances - min_distances.min()) / (min_distances.max() - min_distances.min())

# ساخت سری وزن ناهنجاری برای همه داده‌ها
anomaly_weights = np.zeros(len(df))
anomaly_weights[noise_mask.values] = normalized_weights

# اضافه کردن به دیتافریم نهایی
df['anomaly_weight'] = anomaly_weights
